# Day 5 Lab - Write it yourself: an RNN for heartbeats

This morning you *changed* numbers in the code breaks. Now you **write the code yourself** - step by step:
- **Part 1 - Look at the data** (about 10 min): ECG heartbeats, shapes for an RNN
- **Part 2 - Your RNN classifier** (about 20 min): one recurrent layer + one linear layer
- **Part 3 - Train it** (about 15 min): accuracy vs balanced accuracy, class weights
- **Part 4 - RNN vs GRU vs LSTM vs CNN** (about 20 min): one table, four models
- **⭐ Bonus** (homework): bidirectional, a bigger memory, the test set

What is not finished in class is homework. A **CPU runtime is enough**.

How the writing tasks work:
1. read the **plan** (a small table),
2. write your code below the numbered steps - where it says `pass  # YOUR CODE HERE`,
3. stuck? open **💡 Hint**,
4. run the **check cell**: ✅ = correct, ❌ = it tells you what is wrong.

You cannot break anything - if something looks strange: *Runtime → Restart session*, then run the setup cells again.

## Setup - run these two cells first
The first cell loads **ECG5000**: 5 000 heartbeats from one long ECG recording, each cut to **140 time steps** and
already normalised (mean 0, spread 1 per beat). Every beat has one of 5 labels:

| class | name | what it is |
|---|---|---|
| 0 | normal | a normal heartbeat |
| 1 | R-on-T PVC | an extra beat from the ventricles that falls on the T-wave of the beat before |
| 2 | PVC | premature ventricular contraction - an extra beat from the ventricles |
| 3 | SP | supraventricular premature beat - an extra beat from above the ventricles |
| 4 | UB | unclassified beat |

We split the beats **70 / 15 / 15 %** into training, validation and test - *stratified*: every part gets the same
share of every class.

In [ ]:
import copy
import io
import time
import urllib.request
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.metrics import ConfusionMatrixDisplay, balanced_accuracy_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset


def load_ecg5000(root="data/ecg5000"):
    """All 5000 beats: X (5000, 140, 1) = (beats, time steps, features), y (5000,) = class 0...4."""
    root = Path(root)
    if not (root / "ECG5000_TEST.txt").exists():
        url = "https://www.timeseriesclassification.com/aeon-toolkit/ECG5000.zip"
        zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url).read())).extractall(root)
    rows = [[float(v) for v in line.split()] for name in ("ECG5000_TRAIN.txt", "ECG5000_TEST.txt")
            for line in (root / name).read_text().splitlines() if line.strip()]
    data = torch.tensor(rows)
    return data[:, 1:].unsqueeze(-1), data[:, 0].long() - 1


X, y = load_ecg5000()
idx_train, idx_rest = train_test_split(torch.arange(len(y)), test_size=0.3, stratify=y, random_state=0)
idx_val, idx_test = train_test_split(idx_rest, test_size=0.5, stratify=y[idx_rest], random_state=0)
X_train, y_train = X[idx_train], y[idx_train]       # training beats
X_val, y_val = X[idx_val], y[idx_val]               # validation beats: for choosing
X_test, y_test = X[idx_test], y[idx_test]           # test beats: only at the very end
print("training:", tuple(X_train.shape), "  validation:", tuple(X_val.shape), "  test:", tuple(X_test.shape))

The second cell defines the **check cells** and other helpers for the whole lab.
**Just run it - you do not need to read it.**

In [ ]:
NAMES = ["normal", "R-on-T PVC", "PVC", "SP", "UB"]


def show_beats():
    fig, axes = plt.subplots(1, 5, figsize=(16, 3), sharey=True)
    for c, ax in enumerate(axes):
        ax.plot(X_train[y_train == c][:20, :, 0].T, color="#2E86AB", alpha=0.25)
        ax.set_title(f"{c}: {NAMES[c]}  ({int((y_train == c).sum())} beats)")
        ax.set_xlabel("time step")
    plt.tight_layout()
    plt.show()


def n_params(model):
    return sum(p.numel() for p in model.parameters())


def check_model(ECGRNN):
    expected = {("RNN", False): 1285, ("GRU", False): 3525, ("LSTM", False): 4645, ("GRU", True): 7045}
    for (cell_type, bidirectional), n_expected in expected.items():
        name = cell_type + (" bidirectional" if bidirectional else "")
        try:
            model = ECGRNN(cell_type, bidirectional=bidirectional)
            if model.rnn is None or model.out is None:
                print("❌ self.rnn or self.out is still None - write steps 1-3 in __init__")
                return
            out = model(torch.zeros(8, 140, 1))
        except Exception as e:
            print("❌", name, "- error:", type(e).__name__, str(e).split("\n")[0])
            return
        if out is None:
            print("❌ forward() returns None - did you write `return ...`?")
            return
        if tuple(out.shape) != (8, 5):
            print("❌", name, "- the output should be (8, 5): one score per class - yours is", tuple(out.shape))
            return
        if n_params(model) != n_expected:
            hint = "Linear(size, 5)? bidirectional=bidirectional in the layer?" if bidirectional else "hidden=32? Linear(size, 5)?"
            print("❌", name, "- should have", n_expected, "parameters, yours has", n_params(model), "-", hint)
            return
        print("✅", name.ljust(18), "output", tuple(out.shape), "  parameters:", n_params(model))


def class_weights():
    """Rare classes count more: weight = (number of beats) / (5 · number of beats of this class)."""
    counts = torch.bincount(y_train, minlength=5).float()
    return len(y_train) / (5 * counts)


def predict(model, X):
    model.eval()
    with torch.no_grad():
        return model(X).argmax(1)


def train_model(model, epochs=20, lr=0.003, weighted=False):
    """Train on the training beats (Adam, batch 64, gradient clipping) and return the validation scores."""
    torch.manual_seed(0)
    loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights() if weighted else None)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    start = time.time()
    for epoch in range(epochs):
        model.train()
        for x, y in loader:
            loss = loss_fn(model(x), y)
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)      # against exploding gradients (block 1)
            opt.step()
    pred = predict(model, X_val)
    scores = {"accuracy": round((pred == y_val).float().mean().item(), 3),
              "balanced accuracy": round(balanced_accuracy_score(y_val, pred), 3),
              "parameters": n_params(model), "seconds": round(time.time() - start)}
    print(scores)
    return scores


def show_confusion(model, X=None, y=None):
    X, y = (X_val, y_val) if X is None else (X, y)
    ConfusionMatrixDisplay.from_predictions(y, predict(model, X), display_labels=NAMES, xticks_rotation=30,
                                            colorbar=False)
    plt.title("rows: true class - columns: predicted class")
    plt.show()


class ECGCNN(nn.Module):
    """A small 1D CNN as on day 4 (given): two Conv1d blocks, then the average over time -> 5 scores."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=7, padding=3), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(16, 32, kernel_size=7, padding=3), nn.ReLU(), nn.AdaptiveAvgPool1d(1),
            nn.Flatten(), nn.Linear(32, 5))

    def forward(self, x):                        # x: (batch, time, 1)
        return self.net(x.transpose(1, 2))      # Conv1d wants (batch, channels, time)


def check_results(results):
    missing = [k for k in ["RNN", "GRU", "LSTM", "CNN"] if k not in results]
    if missing:
        print("❌ still missing:", missing)
        return
    print(f"{'model':6s} {'accuracy':>9s} {'balanced':>9s} {'parameters':>11s} {'seconds':>8s}")
    for name, s in results.items():
        print(f"{name:6s} {s['accuracy']:9.3f} {s['balanced accuracy']:9.3f} {s['parameters']:11d} {s['seconds']:8d}")
    best = max(results, key=lambda k: results[k]["balanced accuracy"])
    print("✅ all four trained - best balanced accuracy:", best)


print("helpers ready")

# Part 1 - Look at the data (about 10 min)
## 1.1 Five kinds of heartbeats
Every panel shows 20 training beats of one class.

In [ ]:
show_beats()

print(X.shape)

❓ Which classes look **similar**? How many training beats do the two rarest classes have?

*Your answer:*

<details><summary>👀 What you should see</summary>

- normal beats start with a deep dip, then a slow wave and a peak at about step 100-110
- R-on-T PVC and SP look similar (a dip at the start and a deep dip at the end) - they will be confused
- very **imbalanced**: 2 043 normal, 1 237 R-on-T PVC, but only **67** PVC, 136 SP and **17** UB beats -
  like the skin lesions in the challenge
</details>

## 1.2 Shapes for an RNN
An RNN layer (`nn.RNN`, `nn.GRU`, `nn.LSTM` with `batch_first=True`) gets the data as a block of numbers with
**three** sizes, always in this order: **(batch, time steps, features)**. Run the cell and look at the shapes.

In [ ]:
print("a batch of 8 beats:", tuple(X_train[:8].shape))
print("one single beat:   ", tuple(X_train[0].shape))

❓ What does each of the three numbers of `(8, 140, 1)` mean? Why is the last number 1 - and what would it be for an
EEG with 2 channels?

*Your answer:*

<details><summary>👀 What you should see</summary>

- `(8, 140, 1)` = **8** beats (batch) · **140** values per beat, one after the other (time steps) · **1** number per
  time step (features)
- the last number is 1 because we have **one** ECG lead. An EEG with 2 channels: `(8, T, 2)`; a 12-lead ECG: 12
- one single beat is only `(140, 1)` - the batch size is missing. To give one beat to the RNN, use `X_train[0:1]`
  (shape `(1, 140, 1)`)
</details>

# Part 2 - Your RNN classifier (about 20 min)
## 2.1 The plan
A **many-to-one** model: the recurrent layer reads the beat step by step, then a linear layer turns the **last**
memory into 5 class scores.

| step | layer / code | in | out | shape |
|---|---|---|---|---|
| 1 | `nn.RNN`, `nn.GRU` or `nn.LSTM` (depends on `cell_type`) | 1 number per step | `hidden` memory numbers | (8, 140, 1) → (8, 140, 32) |
| 2 | take the memory after the **last** time step: `out[:, -1]` | | | (8, 140, 32) → (8, 32) |
| 3 | `nn.Linear(size, 5)` | 32 | 5 | (8, 32) → (8, 5) |

`cell_type` is the text `"RNN"`, `"GRU"` or `"LSTM"` - so the **same class** builds all three models:
`ECGRNN("GRU")`. With `bidirectional=True` the layer also reads the beat backwards (block 2) - then it gives
2 × 32 numbers per step, so `size` is 64.

✏️ **Write `ECGRNN`:** steps 1-3 in `__init__`, the forward pass in `forward`.

In [ ]:
class ECGRNN(nn.Module):
    def __init__(self, cell_type="GRU", hidden=32, bidirectional=False):
        super().__init__()
        torch.manual_seed(0)     # same start weights for everybody
        self.rnn = None
        self.out = None
        size = 2 * hidden if bidirectional else hidden      # numbers per time step that come out of the layer
        # 1. if cell_type == "RNN":
        #        self.rnn = nn.RNN(input_size=..., hidden_size=..., batch_first=..., bidirectional=...)
        # 2. elif cell_type == "GRU":   the same with nn.GRU
        #    elif cell_type == "LSTM":  the same with nn.LSTM
        # 3. self.out = nn.Linear(size, 5)
        pass  # YOUR CODE HERE

    def forward(self, x):
        scores = None
        # 4. out, _ = self.rnn(x)            (the _ is the last memory - the LSTM gives two, so we ignore it)
        # 5. scores = self.out( the last time step of out )
        pass  # YOUR CODE HERE
        return scores

<details><summary>💡 Hint</summary>

- step 1: `self.rnn = nn.RNN(input_size=1, hidden_size=hidden, batch_first=True, bidirectional=bidirectional)` -
  1 number per time step, the memory size `hidden`, data as (batch, time, features), and the switch `bidirectional`
- step 2: copy the line of step 1 two more times and change only the two words: `"GRU"` / `nn.GRU`,
  `"LSTM"` / `nn.LSTM`. Careful: `elif`, and the `self.rnn = ...` line is indented 4 spaces more than `if`
- step 3: `self.out = nn.Linear(size, 5)`
- step 5: `out` has the shape (batch, time, size); `out[:, -1]` = all beats, last time step
- `out, h = self.rnn(x)` fails for nobody, but for the LSTM `h` is a pair (h, c) - so we write `out, _`
</details>

In [ ]:
check_model(ECGRNN)

❓ The RNN has **1 285** parameters, the LSTM **4 645**. Why does the number **not** depend on the 140 time steps?

*Your answer:*

<details><summary>👀 What you should see</summary>

- ✅ RNN **1 285**, GRU **3 525**, LSTM **4 645** parameters, output (8, 5) for all three
- ✅ GRU bidirectional **7 045** - two GRUs (2 × 3 360) and a Linear layer with 64 inputs (325)
- RNN: 32·1 + 32·32 + 2·32 = 1 120 in the recurrent layer + 32·5 + 5 = 165 in the linear layer.
  GRU ≈ 3 ×, LSTM ≈ 4 × the recurrent part (gates)
- the **same** weights are used at every time step - a beat of 1 000 steps would need no new weights
</details>

# Part 3 - Train it (about 15 min)
## 3.1 A GRU - and two scores
`train_model(model, epochs, weighted)` is given (helper cell): the training loop of day 3, with gradient clipping.
It prints the **accuracy** and the **balanced accuracy** (the mean of the sensitivity per class - the main metric
of the challenge) on the validation beats. About 30-60 s on a CPU.

In [ ]:
EPOCHS = 20      # 👈

gru = ECGRNN("GRU")
if gru.rnn is not None:
    scores_gru = train_model(gru, epochs=EPOCHS)
    show_confusion(gru)

❓ **Explain in one sentence:** why is the accuracy so much higher than the balanced accuracy? Look at the rows of
the rare classes in the confusion matrix.

*Your answer:*

<details><summary>👀 What you should see</summary>

- accuracy **0.95**, balanced accuracy only **0.53** (about 45 s on a CPU)
- normal and R-on-T PVC are almost always right - but only 5 of 14 PVC, 8 of 29 SP and 0 of 4 UB beats are found;
  the rest is predicted as normal or R-on-T PVC
- answer: 88 % of the beats are normal or R-on-T PVC, so a model that gets only these two right already has a high
  accuracy. The balanced accuracy gives every class the same weight - it shows that the rare classes are missed.
</details>

## 3.2 Class weights
`weighted=True` makes mistakes on rare classes more expensive in the loss: a UB beat counts about **120 ×** more
than a normal beat (weight = number of beats / (5 · beats of this class), see `class_weights()`).

🔧 Run it and compare with 3.1.

In [ ]:
print("class weights:", [round(w, 2) for w in class_weights().tolist()])
gru_w = ECGRNN("GRU")
if gru_w.rnn is not None:
    scores_gru_w = train_model(gru_w, epochs=EPOCHS, weighted=True)
    show_confusion(gru_w)

❓ **Explain in one sentence:** what happens to accuracy and balanced accuracy - and is that a good trade?

*Your answer:*

<details><summary>👀 What you should see</summary>

- class weights **0.34, 0.57, 10.4, 5.1, 41.2**
- accuracy goes **down** (0.95 → **0.88**), balanced accuracy goes **up** (0.53 → **0.70**)
- more rare beats are found now: PVC 7 of 14, SP 14 of 29, UB 3 of 4 - but about 60 R-on-T PVC beats are now
  wrongly called PVC or SP
- answer: in medicine missing the rare abnormal beats is usually worse than a few false alarms, so yes. Which trade
  is right depends on the task - this is why we decide on the metric *before* we train (day 3).
- the numbers jump a bit with the start weights: there are only **4** UB beats in the validation set
</details>

# Part 4 - RNN vs GRU vs LSTM vs CNN (about 20 min)
## 4.1 One table, four models
✏️ **Write the loop:** train every model in `models` with class weights and store its scores in `results`.
The CNN is the day-4 1D CNN (given in the helper cell). All four take 1-2 minutes together.

In [ ]:
models = {
    "RNN": ECGRNN("RNN"),
    "GRU": ECGRNN("GRU"),
    "LSTM": ECGRNN("LSTM"),
    "CNN": ECGCNN(),
}
results = {}
# 1. for name, model in models.items():
# 2.     print(name)
# 3.     results[name] = train_model( ... the model, EPOCHS, with class weights ... )
pass  # YOUR CODE HERE
check_results(results)

<details><summary>💡 Hint</summary>

- `models.items()` gives pairs (name, model) - like `("GRU", ECGRNN("GRU"))`
- step 3: `results[name] = train_model(model, epochs=EPOCHS, weighted=True)`
</details>

❓ **Explain in two sentences:** which model is best on the balanced accuracy? Which one is fastest - and why is
the plain RNN worse than GRU and LSTM, although the beats are only 140 steps long?

*Your answer:*

<details><summary>👀 What you should see</summary>

| model | accuracy | balanced accuracy | parameters | seconds (CPU) |
|---|---|---|---|---|
| RNN | 0.77 | 0.45 | 1 285 | 12 |
| GRU | 0.88 | 0.70 | 3 525 | 41 |
| LSTM | 0.88 | 0.58 | 4 645 | 16 |
| CNN | 0.72 | 0.47 | 3 909 | 4 |

- GRU and LSTM are clearly better than the plain RNN - the important part of a beat (the start) is 140 steps away
  from the readout at the end; the plain RNN has forgotten most of it (vanishing gradient, block 1)
- the CNN is by far the fastest: all time steps are computed at once. An RNN must go step by step. This small CNN
  is weaker here - with more filters or epochs it catches up (try it).
- GRU vs LSTM: here the GRU wins, with another seed it can be the LSTM - in big studies there is no clear winner
- your numbers can differ by a few points (other computer, other PyTorch version) - only 4 UB and 14 PVC beats
  decide a large part of the balanced accuracy. A fair comparison needs several seeds (day 3).
</details>

## 4.2 Where does the best model fail?
🔧 Choose your best model from the table and look at its confusion matrix.

In [ ]:
BEST = "GRU"      # 👈 the name of your best model

if BEST in results:
    show_confusion(models[BEST])

❓ Which two classes are confused most often? Look at the plots of Part 1 - why?

*Your answer:*

<details><summary>👀 What you should see</summary>

- GRU: most mistakes are **R-on-T PVC → SP** (33) and **R-on-T PVC → PVC** (24), and **SP → R-on-T PVC** (10)
- SP and R-on-T PVC have almost the same shape in the plots of 1.1 (dip at the start, deep dip at the end). The
  class weights push the model towards the rare classes, so now the big class R-on-T PVC pays the price
</details>

# ⭐ Bonus (homework)
1. **Bidirectional:** train `ECGRNN("GRU", bidirectional=True)` with class weights. Is it better than the GRU?
   Why does the Linear layer now need 64 inputs? (Hint: code break 3 of block 2.)
2. **A bigger memory:** `ECGRNN("LSTM", hidden=64)`. More parameters - better?
3. **The test set - once, at the very end:** choose your final model on the validation scores, then call
   `show_confusion(model, X_test, y_test)` and compute its balanced accuracy on the test beats. Never go back and
   change the model after this (day 3).

In [ ]:
# your bonus code here
pass  # YOUR CODE HERE

<details><summary>👀 What you should see</summary>

- bidirectional GRU: the Linear layer gets 64 inputs (`size`), because `out` now has 2 × 32 numbers per step; **7 045**
  parameters, balanced accuracy **0.67** (GRU: 0.70) - for 140-step beats the backward direction adds nothing here,
  and it takes twice as long (about 90 s)
- the GRU on the test beats: balanced accuracy **0.66** (validation 0.70) - close, as it should be. With only
  3 UB beats in the test set, one beat changes the score by 0.07
</details>

# For your project
- **sequences in PyTorch:** `(batch, time steps, features)` with `batch_first=True`; `out[:, -1]` = many-to-one
- **try GRU and LSTM** - it is one word; the plain RNN is only a baseline
- **gradient clipping** (`clip_grad_norm_` between `backward()` and `step()`) is cheap insurance for every RNN
- **imbalanced classes:** balanced accuracy + confusion matrix, never accuracy alone; class weights in the loss
  are one line (`nn.CrossEntropyLoss(weight=...)`) - the challenge has the same problem
- **a CNN is often a strong and fast baseline** for short signals - compare with it before you build something big